In [12]:
from src.photometry.data_preprocessing import *
from src.photometry.data_import import *
from src.behavior.data_import import *
from src.behavior.data_preprocessing import *
from typing import Dict

In [18]:
mice = ["1140654", "1140657", "1140703", "1140704"]
for mouse in mice:
    #assess_photometry_reward_alignment(mouse)
    #assess_photometry_reward_alignment(mouse, sample_rate=130, pre_time=2.5, post_time=2.5, zoom_range=(15000, 15000+1300*6))
    analyze_photometry_reward_alignment_double(mouse, sample_rate=130, pre_time=5, post_time=5)

Processing mouse 1140654...
Processing day27_record...
Using folder path: public\projects\MeAh_16062021_Photometry\ExpExp\x-Meg\training_3bandit_with poke\1140654\day27_record
Processed file: public\projects\MeAh_16062021_Photometry\ExpExp\x-Meg\training_3bandit_with poke\1140654\day27_record\day27-2025-04-17-073942.txt
No behavior match found for day28, skipping.
No behavior match found for day29, skipping.
No behavior match found for day30, skipping.
No behavior match found for day31, skipping.
No behavior match found for day32, skipping.
Analyzing session: day27
     n_trials  n_rewards  total_reward_amount  n_state_changes  \
0           1          1                  3.5                0   
1           2          1                  3.5                0   
2           3          2                  7.0                0   
3           4          2                  7.0                0   
4           5          3                 10.5                0   
..        ...        ...        

KeyError: ['poke_to_outcome']

In [2]:
def analyze_photometry_reward_alignment(mouse_id, sample_rate=130, pre_time=0.1, post_time=0.25):
    print(f"Processing mouse {mouse_id}...")

    for recording_day in get_recording_days(mouse_id):
        print(f"Processing {recording_day}...")

        # 1. Load and preprocess photometry data
        photometry_sessions = load_photometry_data(mouse_id)
        for i in range(len(photometry_sessions)):
            photometry_sessions[i]["preprocessed_data"] = preprocess_photometry(photometry_sessions[i]["data"])

        # 2. Load behavior data
        behavior_sessions = import_behavior(mouse_id, recording_day)

        # 3. Align behavior to photometry
        aligned_sessions = align_photometry_to_behavior(
            photometry_sessions, behavior_sessions,
            sample_rate=sample_rate,
            pre_time=pre_time,
            post_time=post_time
        )

        # 4. Loop over each aligned session and analyze
        for aligned_session in aligned_sessions:
            day = aligned_session['day']
            print(f"Analyzing session: {day}")

            # Match to behavior session
            matches = [b for b in behavior_sessions if b['day'] == day]
            if not matches:
                print(f"No behavior match found for {day}, skipping.")
                continue

            b_session = matches[0]

            # 5. Split by reward
            trial_split = split_trials_by_reward(aligned_session, b_session)

            # 6. Compute stats
            reward_stats = compute_mean_sem(trial_split['rewarded_trials'])
            no_reward_stats = compute_mean_sem(trial_split['unrewarded_trials'])

            # 7. Plot
            time_axis = np.linspace(-pre_time, post_time, int((pre_time + post_time) * sample_rate))
            plot_title = f"Mouse {mouse_id} - {day} - Aligned to Outcome"
            plot_rewarded_vs_unrewarded(reward_stats, no_reward_stats, x_axis=time_axis, title=plot_title)

In [3]:
def split_trials_by_reward(aligned_session: Dict, behavior_session: Dict) -> Dict:
    """
    Splits aligned trials in a session into rewarded and unrewarded using the behavior session's outcome field.
    Adds 'rewarded_trials' and 'unrewarded_trials' keys to the aligned_session.
    """
    summary_df = behavior_session["summary"]
    summary_df = summary_df.dropna(subset=["time", "outcome"]).reset_index(drop=True)
    outcomes = summary_df["outcome"].astype(int).values

    aligned_trials = aligned_session["aligned_trials"]
    if aligned_trials.shape[1] != len(outcomes):
        min_len = min(aligned_trials.shape[1], len(outcomes))
        aligned_trials = aligned_trials[:, :min_len]
        outcomes = outcomes[:min_len]

    rewarded = aligned_trials[:, outcomes == 1]
    unrewarded = aligned_trials[:, outcomes == 0]

    return {
        "rewarded_trials": rewarded,
        "unrewarded_trials": unrewarded
    }

def compute_mean_sem(trials: np.ndarray) -> Dict:
    """
    Takes a 2D array of shape (timepoints, n_trials) and returns z-scored trials,
    mean, and SEM across trials.
    """
    mean_trace = trials.mean(axis=1)
    sem_trace = trials.std(axis=1) / np.sqrt(trials.shape[1])
    return {
        "mean": mean_trace,
        "sem": sem_trace
    }

In [4]:
def plot_rewarded_vs_unrewarded(mean_sem_rewarded: Dict, mean_sem_unrewarded: Dict, x_axis: np.ndarray = None, title: str = "Photometry Aligned to Reward"):

    if x_axis is None:
        x_axis = np.linspace(-1, 4, mean_sem_rewarded["mean"].shape[0])  # default fallback
    
    plt.figure(figsize=(8, 5))
    plt.fill_between(x_axis, mean_sem_rewarded["mean"] + mean_sem_rewarded["sem"],
                     mean_sem_rewarded["mean"] - mean_sem_rewarded["sem"],
                     color='green', alpha=0.3)
    plt.plot(x_axis, mean_sem_rewarded["mean"], color='green', label="Rewarded")

    plt.fill_between(x_axis, mean_sem_unrewarded["mean"] + mean_sem_unrewarded["sem"],
                     mean_sem_unrewarded["mean"] - mean_sem_unrewarded["sem"],
                     color='red', alpha=0.3)
    plt.plot(x_axis, mean_sem_unrewarded["mean"], color='red', label="Unrewarded")

    plt.axvspan(0, 0.08, color='royalblue', alpha=0.3, label='Reward Delivery')
    plt.axvline(0, color='black', linestyle='--')

    plt.xlabel("Time from Reward (s)")
    plt.ylabel("Z-scored Signal")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [17]:
def warp_trials_to_fixed_delay(trials: np.ndarray, behavior_session: Dict, fixed_delay: float, sample_rate: int) -> np.ndarray:
    """
    Warps the trials so that the delay between poke and reward is fixed across trials.
    
    trials: (timepoints, n_trials) array aligned to poke.
    behavior_session: dict containing "summary" DataFrame with poke-to-outcome times.
    fixed_delay: desired fixed delay between poke and reward (seconds).
    sample_rate: sampling rate (Hz).
    
    Returns:
    warped_trials: (new_timepoints, n_trials) array
    """
    print(behavior_session["summary"])
    summary_df = behavior_session["summary"].dropna(subset=["time", "outcome", "poke_to_outcome"])
    poke_to_outcome = summary_df["poke_to_outcome"].values  # should be in seconds
    
    warped_trials = []
    for i in range(trials.shape[1]):
        trial = trials[:, i]
        actual_delay = poke_to_outcome[i]
        
        # Determine indexes
        poke_idx = np.argmin(np.abs(np.linspace(-1, 4, trial.shape[0])))  # assuming original time axis from -1 to 4s
        reward_idx = poke_idx + int(actual_delay * sample_rate)

        # Define three parts: pre-poke, between-poke-reward, post-reward
        pre = trial[:poke_idx]
        middle = trial[poke_idx:reward_idx]
        post = trial[reward_idx:]
        
        # Warp middle to match fixed_delay
        target_middle_length = int(fixed_delay * sample_rate)
        warped_middle = np.interp(
            np.linspace(0, len(middle)-1, target_middle_length),
            np.arange(len(middle)),
            middle
        )
        
        warped_trial = np.concatenate([pre, warped_middle, post])
        warped_trials.append(warped_trial)
    
    # Pad to same length (find max length)
    max_len = max(len(trial) for trial in warped_trials)
    padded_trials = np.zeros((max_len, len(warped_trials)))
    padded_trials[:] = np.nan
    for idx, trial in enumerate(warped_trials):
        padded_trials[:len(trial), idx] = trial
    
    return padded_trials

def plot_rewarded_vs_unrewarded_double(
    mean_sem_rewarded_outcome: Dict, mean_sem_unrewarded_outcome: Dict,
    mean_sem_rewarded_poke: Dict, mean_sem_unrewarded_poke: Dict,
    x_axis: np.ndarray, fixed_poke_to_reward_delay: float,
    title: str = "Photometry Aligned to Reward and Poke"):

    fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)

    # Plot outcome-aligned
    ax = axes[0]
    ax.fill_between(x_axis, mean_sem_rewarded_outcome["mean"] + mean_sem_rewarded_outcome["sem"],
                    mean_sem_rewarded_outcome["mean"] - mean_sem_rewarded_outcome["sem"],
                    color='green', alpha=0.3)
    ax.plot(x_axis, mean_sem_rewarded_outcome["mean"], color='green', label="Rewarded")

    ax.fill_between(x_axis, mean_sem_unrewarded_outcome["mean"] + mean_sem_unrewarded_outcome["sem"],
                    mean_sem_unrewarded_outcome["mean"] - mean_sem_unrewarded_outcome["sem"],
                    color='red', alpha=0.3)
    ax.plot(x_axis, mean_sem_unrewarded_outcome["mean"], color='red', label="Unrewarded")

    ax.axvline(0, color='black', linestyle='--', label='Reward')
    ax.set_title("Aligned to Outcome")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Z-scored Signal")
    ax.legend()

    # Plot poke-aligned
    ax = axes[1]
    ax.fill_between(x_axis, mean_sem_rewarded_poke["mean"] + mean_sem_rewarded_poke["sem"],
                    mean_sem_rewarded_poke["mean"] - mean_sem_rewarded_poke["sem"],
                    color='green', alpha=0.3)
    ax.plot(x_axis, mean_sem_rewarded_poke["mean"], color='green', label="Rewarded")

    ax.fill_between(x_axis, mean_sem_unrewarded_poke["mean"] + mean_sem_unrewarded_poke["sem"],
                    mean_sem_unrewarded_poke["mean"] - mean_sem_unrewarded_poke["sem"],
                    color='red', alpha=0.3)
    ax.plot(x_axis, mean_sem_unrewarded_poke["mean"], color='red', label="Unrewarded")

    ax.axvline(0, color='purple', linestyle='--', label='Poke')
    ax.axvline(fixed_poke_to_reward_delay, color='black', linestyle='--', label='Reward')
    ax.set_title("Aligned to Poke (Warped)")
    ax.set_xlabel("Time (s)")
    ax.legend()

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

def analyze_photometry_reward_alignment_double(mouse_id, sample_rate=130, pre_time=0.1, post_time=0.25, fixed_poke_to_reward_delay=0.8):
    print(f"Processing mouse {mouse_id}...")

    for recording_day in get_recording_days(mouse_id):
        print(f"Processing {recording_day}...")

        # Load and preprocess photometry data
        photometry_sessions = load_photometry_data(mouse_id)
        for i in range(len(photometry_sessions)):
            photometry_sessions[i]["preprocessed_data"] = preprocess_photometry(photometry_sessions[i]["data"])

        # Load behavior data
        behavior_sessions = import_behavior(mouse_id, recording_day)

        # Align behavior to photometry
        aligned_sessions = align_photometry_to_behavior(
            photometry_sessions, behavior_sessions,
            sample_rate=sample_rate,
            pre_time=pre_time,
            post_time=post_time
        )

        for aligned_session in aligned_sessions:
            day = aligned_session['day']
            print(f"Analyzing session: {day}")

            matches = [b for b in behavior_sessions if b['day'] == day]
            if not matches:
                print(f"No behavior match found for {day}, skipping.")
                continue
            b_session = matches[0]

            # Split trials
            trial_split = split_trials_by_reward(aligned_session, b_session)

            # Reward aligned stats
            reward_stats = compute_mean_sem(trial_split['rewarded_trials'])
            no_reward_stats = compute_mean_sem(trial_split['unrewarded_trials'])

            # Poke aligned (with warping) stats
            warped_trials = warp_trials_to_fixed_delay(aligned_session["aligned_trials"], b_session, fixed_poke_to_reward_delay, sample_rate)
            # you can split warped_trials into rewarded/unrewarded too if needed
            warped_rewarded = warped_trials[:, b_session["summary"]["outcome"].values == 1]
            warped_unrewarded = warped_trials[:, b_session["summary"]["outcome"].values == 0]
            warped_reward_stats = compute_mean_sem(warped_rewarded)
            warped_no_reward_stats = compute_mean_sem(warped_unrewarded)

            # Plotting
            time_axis = np.linspace(-pre_time, fixed_poke_to_reward_delay + post_time, warped_trials.shape[0])
            plot_title = f"Mouse {mouse_id} - {day}"
            plot_rewarded_vs_unrewarded_double(reward_stats, no_reward_stats,
                                               warped_reward_stats, warped_no_reward_stats,
                                               x_axis=time_axis, title=plot_title)


